# Clase 1 — Introducción a Deep Learning con Python y PyTorch

Curso **"Deep Learning con Python y PyTorch desde cero"** — DIH DATAlife, a partir de la certificación EITC/AI/DLPP (European IT Certificate / Artificial Intelligence / Deep Learning with Python and PyTorch).

**Duración estimada de la clase: 90 min**

## Objetivos de aprendizaje de esta clase

- Explicar qué es el aprendizaje profundo (DL, Deep Learning) y en qué se diferencia del aprendizaje automático (ML, Machine Learning) tradicional
- Describir qué es una red neuronal (NN, Neural Network) y por qué PyTorch representa todos los datos como tensores
- Crear y manipular tensores básicos con PyTorch
- Construir una red neuronal mínima con `torch.nn` y calcular su primer gradiente con autograd

Este notebook acompaña al capítulo 01 del manual del curso. Ejecuta cada celda en orden con **Mayús + Intro**.

## Parte 1 — Aprendizaje profundo y redes neuronales

### Aprendizaje profundo frente a aprendizaje automático tradicional

El aprendizaje automático (ML) ya invierte la lógica de la programación tradicional: en vez de escribir reglas a mano, le das al sistema ejemplos y dejas que deduzca el patrón. El aprendizaje profundo (DL) es un subcampo del ML que usa redes neuronales con varias capas ocultas apiladas, capaces de aprender por sí mismas qué características (features) de los datos importan, en lugar de que tú las definas a mano.

Es la diferencia entre seguir una receta ya perfeccionada por otra persona (ML tradicional, donde tú eliges las características) y dejar que un aprendiz pruebe miles de combinaciones hasta encontrar su propia receta (DL, donde el modelo aprende las características). Esta capacidad es la que ha hecho del DL la técnica dominante en visión por computador, procesamiento de lenguaje natural y reconocimiento de voz.

### Qué es una red neuronal

Una red neuronal (NN) es un modelo matemático inspirado, de forma simplificada, en cómo se conectan las neuronas del cerebro. Se organiza en capas:

- **Capa de entrada**: recibe los datos ya convertidos en números
- **Capas ocultas**: capas intermedias sobre las que no tienes control directo
- **Capa de salida**: produce la predicción final

Cada neurona multiplica sus entradas por unos pesos, suma un sesgo (bias) y pasa el resultado por una **función de activación**, que introduce la no linealidad necesaria para aprender patrones complejos. Dos funciones habituales son ReLU (Rectified Linear Unit), que anula los valores negativos y deja pasar los positivos, y sigmoid, que comprime cualquier valor a un rango entre cero y uno.

Puedes ver su forma con un poco de código, aunque todavía no forme parte de ninguna red:

In [ ]:
import torch
import matplotlib.pyplot as plt

x = torch.linspace(-5, 5, 100)
relu = torch.relu(x)
sigmoide = torch.sigmoid(x)

plt.figure(figsize=(6, 4))
plt.plot(x, relu, label="ReLU")
plt.plot(x, sigmoide, label="sigmoid")
plt.axhline(0, color="gray", linewidth=0.5)
plt.legend()
plt.title("Dos funciones de activación habituales")
plt.show()

## Parte 2 — Tensores y autograd en PyTorch

### Qué es un tensor

PyTorch representa absolutamente todo —un número suelto, una imagen, un lote de miles de imágenes— con una única estructura: el **tensor**, un array multidimensional similar a un `ndarray` de NumPy, con la ventaja de poder ejecutarse en una GPU (Graphics Processing Unit). Vamos a comprobarlo con código.

In [ ]:
print("Versión de PyTorch:", torch.__version__)

# Un tensor de una dimensión (vector), a partir de una lista de Python
x = torch.tensor([5, 3])
y = torch.tensor([2, 1])

# PyTorch sobrecarga los operadores habituales: esto multiplica elemento a elemento
print("x * y =", x * y)

In [ ]:
# torch.zeros crea un tensor lleno de ceros con la forma (shape) que le indiques
z = torch.zeros((2, 5))
print("z:\n", z)
print("shape:", z.shape)

# view reorganiza los mismos datos en una forma distinta, sin copiarlos
z_reordenado = z.view(1, 10)
print("shape tras view(1, 10):", z_reordenado.shape)

### Autograd: tu primer gradiente

Entrenar una red significa calcular, para cada peso, cuánto contribuyó al error final: el gradiente. **Autograd** es el módulo de diferenciación automática de PyTorch: registra en un grafo cada operación sobre un tensor marcado con `requires_grad=True` y, al llamar a `.backward()`, recorre ese grafo hacia atrás para calcular los gradientes automáticamente.

In [ ]:
# requires_grad=True le dice a PyTorch que registre cada operación sobre x
x = torch.tensor([2.0, 3.0], requires_grad=True)

y = x + 2
z = y * y * 3

# backward() recorre el grafo hacia atrás y calcula el gradiente de z respecto a x
z.backward(torch.tensor([1.0, 1.0]))

print("x.grad =", x.grad)

No necesitas resolver esta derivada a mano para confiar en el resultado: lo importante es que `.backward()` dispara todo el cálculo de gradientes y que `x.grad` es donde PyTorch los deja disponibles.

### Una red neuronal mínima con `nn.Module` y `nn.Linear`

Toda red neuronal en PyTorch sigue el mismo patrón: una clase que hereda de `nn.Module`, con sus capas declaradas en `__init__` y la lógica del forward pass en el método `forward`.

In [ ]:
import torch.nn as nn

class RedMinima(nn.Module):
    def __init__(self):
        super().__init__()
        # nn.Linear(3, 1): una capa totalmente conectada que recibe tres
        # valores de entrada y produce un único valor de salida
        self.capa = nn.Linear(3, 1)

    def forward(self, x):
        return self.capa(x)

modelo = RedMinima()
print(modelo)

In [ ]:
# Un único ejemplo de entrada, con tres características
entrada = torch.tensor([[1.0, 2.0, 3.0]])

# El forward pass: PyTorch llama a forward() automáticamente al usar modelo(entrada)
salida = modelo(entrada)
print("Salida del modelo:", salida)

In [ ]:
# Como salida depende de pesos con requires_grad=True (por defecto en nn.Linear),
# podemos propagar el error hacia atrás y obtener el gradiente de cada peso
salida.backward()

print("Gradiente de los pesos:", modelo.capa.weight.grad)
print("Gradiente del sesgo:", modelo.capa.bias.grad)

## Resumen de la clase

- El aprendizaje profundo es un subcampo del ML que usa redes con varias capas para aprender representaciones de los datos, sin que tú definas a mano qué características importan
- Una red neuronal se organiza en capas de entrada, ocultas y salida; cada conexión tiene un peso, y una función de activación introduce la no linealidad necesaria para aprender patrones complejos
- Un tensor es la estructura central de PyTorch: un array multidimensional que representa cualquier dato y puede ejecutarse en CPU o en GPU
- Autograd calcula automáticamente los gradientes de los tensores con `requires_grad=True` cuando llamas a `.backward()`
- En PyTorch, una red se define como una clase que hereda de `nn.Module`, con capas como `nn.Linear` en `__init__` y la lógica del forward pass en `forward`

En el notebook de práctica vas a crear tus propios tensores y tu propia red de una neurona.